# Bronze Layer — Raw Data Ingestion

This notebook implements the Bronze layer of the Medallion Architecture for the Retail Lakehouse project in Databricks.

It ingests raw Olist e-commerce CSV files from Unity Catalog Volumes into Delta tables in the `retail_lakehouse.bronze` schema.

## What This Notebook Does

* Configuration-driven ingestion framework that loads all 9 source tables
* A single reusable ingestion function reads each file, adds metadata, and writes to Delta
* A loop runs every table and reports success/failure at the end
* Error isolation ensures one table's failure does not stop the others

## Input

Raw CSV files stored in `/Volumes/retail_lakehouse/bronze/retail_data/`

## Output

9 Delta tables in the `retail_lakehouse.bronze` schema

## Design Decisions

* **Explicit schemas instead of inference** — for reproducibility and correct typing; inference can misread identifiers and adds an extra read pass
* **Identifier columns (zip codes, IDs) typed as strings** — to preserve leading zeros and exact format
* **Overwrite mode** — keeps the load idempotent: re-running reloads the full static source without creating duplicates
* **Lineage metadata columns** (`ingestion_time`, `source_file`) — added for traceability and auditing
* **Known data issues documented for Silver** — e.g. the geolocation table has non-unique zip-code keys, which would cause row fan-out if joined directly; this is handled in the Silver layer

This layer serves as the foundation for downstream Silver and Gold transformations.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### Storing Catalog and Schema name in variables for reusability.

In [0]:
catalog_name = 'retail_lakehouse' 
schema_name = 'bronze'

### SCHEMA Definition for all 9 tables

In [0]:
config = {"customers": 
                {"source_file_path":'/Volumes/retail_lakehouse/bronze/retail_data/Dataset/olist_customers_dataset.csv',
                 "target_table_name":"customers",
                 "schema": StructType([StructField('customer_id',StringType(),True), 
                    StructField('customer_unique_id',StringType(),True),
                    StructField('customer_zip_code_prefix',StringType(),True),
                    StructField('customer_city',StringType(),True),
                    StructField('customer_state',StringType(),True)])
                 },
           "orders": 
                {"source_file_path": "/Volumes/retail_lakehouse/bronze/retail_data/Dataset/olist_orders_dataset.csv",
                 "target_table_name": "orders",
                 "schema": StructType([StructField('order_id',StringType(),True), 
                    StructField('customer_id',StringType(),True),
                    StructField('order_status',StringType(),True),
                    StructField('order_purchase_timestamp',TimestampType(),True),
                    StructField('order_approved_at',TimestampType(),True),
                    StructField('order_delivered_carrier_date',TimestampType(),True),
                    StructField('order_delivered_customer_date',TimestampType(),True),
                    StructField('order_estimated_delivery_date',TimestampType(),True)])
                 },
            "order_payments":
                 {"source_file_path": "/Volumes/retail_lakehouse/bronze/retail_data/Dataset/olist_order_payments_dataset.csv",
                  "target_table_name": "order_payments",
                  "schema": StructType([StructField('order_id',StringType(),True), 
                    StructField('payment_sequential',IntegerType(),True),
                    StructField('payment_type',StringType(),True),
                    StructField('payment_installments',IntegerType(),True),
                    StructField('payment_value',DecimalType(10,2),True)])
                  },
            "products":
               {"source_file_path": "/Volumes/retail_lakehouse/bronze/retail_data/Dataset/olist_products_dataset.csv",
                "target_table_name": "products",
                "schema": StructType([StructField('product_id',StringType(),True), 
                    StructField('product_category_name',StringType(),True),
                    StructField('product_name_lenght',IntegerType(),True),
                    StructField('product_description_lenght',IntegerType(),True),
                    StructField('product_photos_qty',IntegerType(),True),
                    StructField('product_weight_g',IntegerType(),True),
                    StructField('product_length_cm',IntegerType(),True),
                    StructField('product_height_cm',IntegerType(),True),
                    StructField('product_width_cm',IntegerType(),True)])
               },
            "products_category":
               {"source_file_path": "/Volumes/retail_lakehouse/bronze/retail_data/Dataset/product_category_name_translation.csv",
                "target_table_name": "products_category",
                "schema": StructType([StructField('product_category_name',StringType(),True), 
                    StructField('product_category_name_english',StringType(),True)])
                },
            "orders_items":
               {"source_file_path": "/Volumes/retail_lakehouse/bronze/retail_data/Dataset/olist_order_items_dataset.csv",
                "target_table_name": "orders_items",
                "schema": StructType([StructField('order_id',StringType(),True), 
                    StructField('order_item_id',IntegerType(),True),
                    StructField('product_id',StringType(),True),
                    StructField('seller_id',StringType(),True),
                    StructField('shipping_limit_date',TimestampType(),True),
                    StructField('price',DecimalType(10,2),True),
                    StructField('freight_value',DecimalType(10,2),True)])
                },
            "order_reviews":
               {"source_file_path": "/Volumes/retail_lakehouse/bronze/retail_data/Dataset/olist_order_reviews_dataset.csv",
                "target_table_name": "order_reviews",
                "schema": StructType([StructField('review_id',StringType(),True), 
                    StructField('order_id',StringType(),True),
                    StructField('review_score',IntegerType(),True),
                    StructField('review_comment_title',StringType(),True),
                    StructField('review_comment_message',StringType(),True),
                    StructField('review_creation_date',TimestampType(),True),
                    StructField('review_answer_timestamp',TimestampType(),True)])
                },
            "sellers":
               {"source_file_path": "/Volumes/retail_lakehouse/bronze/retail_data/Dataset/olist_sellers_dataset.csv",
                "target_table_name": "sellers",
                "schema": StructType([StructField('seller_id',StringType(),True), 
                    StructField('seller_zip_code_prefix',StringType(),True),
                    StructField('seller_city',StringType(),True),
                    StructField('seller_state',StringType(),True)])
               },
            "geolocation":
               {"source_file_path": "/Volumes/retail_lakehouse/bronze/retail_data/Dataset/olist_geolocation_dataset.csv",
                "target_table_name": "geolocation",
                "schema": StructType([StructField('geolocation_zip_code_prefix',StringType(),True), 
                    StructField('geolocation_lat',DecimalType(20,10),True),
                    StructField('geolocation_lng',DecimalType(20,10),True),
                    StructField('geolocation_city',StringType(),True),
                    StructField('geolocation_state',StringType(),True)])
               }
}

### ingestion function to ingest the data into Delta table

In [0]:
def ingestion(table_name):
    '''
    ingestion function read the raw file, add audit columns and finally write to Delta tables of bronze layer.
    This function takes the table name as input and returns SUCCESS or FAILURE.
    '''
    try:
        raw_df = spark.read.format('csv').option('header',True).schema(config[table_name]["schema"]).load(config[table_name]["source_file_path"])

        # Adding audit columns for lineage tracking. _metadata.file_name function is used to get the file name of the raw file..
        enriched_df = raw_df.withColumn('ingestion_time',current_timestamp()).withColumn("source_file", col("_metadata.file_name"))

        # Writing the enriched dataframe to delta table. Use overwrite mode to fully load the static file for idempotency. Re-running the code reloads the data without causing duplicates.
        enriched_df.write.mode('overwrite').format('delta').saveAsTable(f'{catalog_name}.{schema_name}.{table_name}')

        record_count = enriched_df.count()
        print(f"{table_name} table loaded successfully: {record_count} records inserted in the {table_name} table")
        return 'SUCCESS'
    
    except Exception as e:
        print(f"Error while loading {table_name} table: {e}")
        return 'FAILURE'

### Calling ingestion Function for each table

In [0]:
success_count = 0
failure_count = 0
succeeded_tables =[]
failed_tables = []

for table_name in config.keys():

    result = ingestion(table_name)

    if result == 'SUCCESS':
        success_count = success_count + 1
        succeeded_tables.append(table_name)

    elif result == 'FAILURE':
        failure_count = failure_count + 1
        failed_tables.append(table_name)

print(f"{success_count} tables loaded successfully and the succeeded tables listed here : {succeeded_tables}")
print(f"{failure_count} tables failed and the failed tables listed here: {failed_tables}")   

customers table loaded successfully: 99441 records inserted in the customers table
orders table loaded successfully: 99441 records inserted in the orders table
order_payments table loaded successfully: 103886 records inserted in the order_payments table
products table loaded successfully: 32951 records inserted in the products table
products_category table loaded successfully: 71 records inserted in the products_category table
orders_items table loaded successfully: 112650 records inserted in the orders_items table
order_reviews table loaded successfully: 104162 records inserted in the order_reviews table
sellers table loaded successfully: 3095 records inserted in the sellers table
geolocation table loaded successfully: 1000163 records inserted in the geolocation table
9 tables loaded successfully and the succeeded tables listed here : ['customers', 'orders', 'order_payments', 'products', 'products_category', 'orders_items', 'order_reviews', 'sellers', 'geolocation']
0 tables failed an